In [1]:
from learn_ai.core.conversation import Conversation
from learn_ai.llm.client import LLMClient
from learn_ai.db.store import ConversationStore


In [2]:
client = LLMClient(provider="groq", model="qwen/qwen3-32b")
store = ConversationStore()
nb = store.create_notebook("New Notebook")

In [ ]:
conv = Conversation(client, store, nb)
conv.send("What did I just ask you?")



'You asked, "What did I just ask you?" \n\nTo clarify: your last message was the same question again. If you\'d like to ask something different or need further assistance, feel free to share! 😊'

### RAG

In [1]:
from learn_ai.rag.loader import load_notes
from learn_ai.rag.chunker import chunk_documents


chunked_documents = chunk_documents(load_notes())

In [2]:
for  chunk in chunked_documents:
    if len(chunk.text) < 500:
        print(f"id: {chunk.id} len: {len(chunk.text)}")

id: 01_neural_networks.md::5 len: 142
id: 02_backpropagation.md::5 len: 172
id: 03_gradient_descent.md::5 len: 113
id: 04_transformers.md::5 len: 184
id: 05_overfitting_regularization.md::5 len: 383
id: 06_probability_distributions.md::5 len: 335
id: 07_hypothesis_testing.md::5 len: 477
id: 07_hypothesis_testing.md::6 len: 27
id: 08_bayesian_inference.md::6 len: 174
id: 09_linear_regression.md::5 len: 417
id: 10_pca.md::6 len: 329


In [1]:
### Test the indexing in the chroma db

from learn_ai.rag.loader import load_notes
from learn_ai.rag.chunker import chunk_documents
from learn_ai.rag.index import index_chunks, get_collection


docs = load_notes()
chunks = chunk_documents(docs)
index_chunks(chunks)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
col = get_collection()
print(f"docs={len(docs)}  chunks={len(chunks)}  in collection={col.count()}")



docs=10  chunks=63  in collection=63


In [6]:
# Cell 2 — peek at what got stored
col.peek(10)   # shows ids, documents, metadatas, embeddings (truncated)



{'ids': ['01_neural_networks.md::0',
  '01_neural_networks.md::1',
  '01_neural_networks.md::2',
  '01_neural_networks.md::3',
  '01_neural_networks.md::4',
  '01_neural_networks.md::5',
  '02_backpropagation.md::0',
  '02_backpropagation.md::1',
  '02_backpropagation.md::2',
  '02_backpropagation.md::3'],
 'embeddings': array([[-1.91796888e-02,  1.62312761e-02,  3.84523459e-02, ...,
         -5.35393265e-05,  2.79254857e-02,  2.28191819e-02],
        [-2.84195282e-02, -1.52176702e-02,  4.33760229e-03, ...,
          2.24235673e-02,  3.04453950e-02, -1.31427143e-02],
        [-4.59317975e-02, -2.64945556e-04,  1.08245797e-02, ...,
         -5.12324739e-03,  3.91896674e-03,  7.81762134e-03],
        ...,
        [-9.89800133e-03, -2.51935655e-03,  2.95171179e-02, ...,
         -3.02057946e-04,  9.62716714e-03, -8.48757382e-03],
        [-1.92475598e-02, -4.31817817e-03,  1.47788664e-02, ...,
          6.35661697e-03,  4.12747897e-02,  8.25680606e-03],
        [-2.35365089e-02, -4.883487

In [47]:
# Cell 3 — first real similarity query (sanity check before building retriever.py)
result = col.query(query_texts=["Deep Learning"], n_results=3)
result

{'ids': [['05_overfitting_regularization.md::2',
   '01_neural_networks.md::4',
   '01_neural_networks.md::1']],
 'embeddings': None,
 'documents': [['n deep learning.\n- **L1 regularization**: adds `λ · Σ |wᵢ|`. Encourages sparsity — many weights become exactly zero. Useful for feature selection.\n- **Elastic net**: a combination of L1 and L2.\n\n## Architectural Regularization\n\n- **Dropout**: during training, randomly zero out a fraction `p` of activations in each layer. Forces the network to learn redundant representations and prevents co-adaptation of neurons. At inference, all units are active and activations are scaled.\n- **Batch normalizati',
   'pact domain, given enough neurons. In practice, **deep** networks (many layers) are far more parameter-efficient than wide shallow networks because they can compose hierarchical features.\n\n## Training\n\nTraining a neural network means finding weight values that minimize a loss function over a dataset. This is done with gradient-ba

In [ ]:
ids = result["ids"][0]
docs = result["documents"][0]
metas = result["metadatas"][0]
distances = result["distances"][0]

In [49]:
ids

['05_overfitting_regularization.md::2',
 '01_neural_networks.md::4',
 '01_neural_networks.md::1']